# GROUP 3

## TEAM

## SETUP

In [2]:
!pip install ftfy regex tqdm torchmetrics
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 18.4 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-wv3gtzim
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-wv3gtzim
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=97779ce4873a3ca0ecf245575205b115229c628f752f0245f13a734379ed3dc2
  Stored in directory: /tmp/pip-ephem-wheel-cache-lscxbbbx/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [3]:
import os

# Define paths
zip_path = './aml-group-project.zip'
extract_to = './AMLdataset'

# Automated Unzip
if not os.path.exists(extract_to):
    os.makedirs(extract_to, exist_ok = True)
    print(f"Unzipping {zip_path}...")

    # Using the shell command is usually faster for large zips
    !unzip -q {zip_path} -d {extract_to}

    print("Unzip complete!")
else:
    print("Dataset already unzipped.")

Unzipping ./aml-group-project.zip...
unzip:  cannot find or open ./aml-group-project.zip, ./aml-group-project.zip.zip or ./aml-group-project.zip.ZIP.
Unzip complete!


## DATASET

In [4]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from PIL import Image
from torch.utils.data import Dataset


class AMLDataset(Dataset):
    def __init__(self, csv_path, imgs_dir, train=True, transform=None):
        self.imgs_dir = imgs_dir
        self.train = train
        self.transform = transform

        full_df = pd.read_csv(csv_path)

        train_df, test_df = train_test_split(
            full_df,
            test_size=0.20,            # 20% for validation
            random_state=42,
            stratify=full_df['label']  # Ensures all 250 classes are in both
        )

        train_df = full_df

        self.df = (train_df if train else test_df).reset_index(drop=True)

        # Create the 'classes' attribute (Unique list of names)
        # We sort them to ensure the index mapping is always consistent
        self.classes = sorted(self.df['label'].unique().tolist())

        # Create a mapping from Name -> Integer ID
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # Create the 'labels' attribute (The ID for every single row)
        # This is helpful if you want to use a Weighted or Balanced Sampler later
        self.labels = [self.class_to_idx[name] for name in self.df['label']]


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_img(row['filename'])
        return img, self.labels[idx]

    def _load_img(self, filename):
        path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

In [5]:
import os
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset


class AMLValidation(Dataset):
    def __init__(self, csv_path, imgs_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.imgs_dir = imgs_dir
        self.transform = transform

        # Group the rows by episode_id so __len__ returns total number of tasks
        self.episode_ids = sorted(self.df['episode_id'].unique())

    def __len__(self):
        return len(self.episode_ids)

    def load_img(self, filename):
        img_path = os.path.join(self.imgs_dir, str(filename))
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        # Return a full 5-way 5-shot Episode
        ep_id = self.episode_ids[idx]
        ep_df = self.df[self.df['episode_id'] == ep_id]

        support_df = ep_df[ep_df['role'] == 'support']
        query_df = ep_df[ep_df['role'] == 'query']

        # Pack support images and their labels
        s_imgs = [f"{self.imgs_dir}/{f}" for f in support_df['filename']]
        s_labels = support_df['label'].values

        # Pack query images
        q_imgs = [f"{self.imgs_dir}/{f}" for f in query_df['filename']]

        return {
            "support_imgs": s_imgs,
            "support_labels": s_labels,
            "query_imgs": q_imgs,
            "episode_id": ep_id
        }

        # Pack support images and their labels
        s_imgs = torch.stack([self.load_img(f) for f in support_df['filename']])
        s_labels = torch.tensor(support_df['label'].values)

        # Pack query images
        q_imgs = torch.stack([self.load_img(f) for f in query_df['filename']])

        # If your test CSV has query labels (sometimes they are hidden), include them:
        # q_labels = torch.tensor(query_df['label'].values) if 'label' in query_df.columns else None

        return {
            "support_imgs": s_imgs,
            "support_labels": s_labels,
            "query_imgs": q_imgs,
            "episode_id": ep_id
        }

In [ ]:
import clip
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load('ViT-B/32', DEVICE, jit=False)
model.eval()
for param in model.parameters():
    param.requires_grad = False

100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 74.3MiB/s]


In [ ]:
from torch.utils.data import DataLoader

# Define constants
LR = 1e-4 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

CSV_PATH  = "./AMLdataset/release/train.csv"
IMGS_DIR  = "./AMLdataset/release/images/"
VAL_CSV   = "./AMLdataset/release/test_episodes_release.csv"

# Download the dataset
train_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=True,
    transform=preprocess
)

test_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=False,
    transform=preprocess
)

val_dataset = AMLValidation(
    csv_path=VAL_CSV,
    imgs_dir=IMGS_DIR,
    transform=preprocess
)

# Prepare the data
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_train = len(train_loader)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_test = len(test_loader)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_val = len(val_loader)

## Zero-Shot Evaluation

In [ ]:
def get_data(dataset):
  images = next(iter(dataset))["support_imgs"]
  texts = next(iter(dataset))["support_labels"]
  queries = next(iter(dataset))["query_imgs"]

  return images, texts, queries

In [ ]:
import clip
import numpy as np

def encode_data(model, preprocess, images: list[str], texts: list[str]):
  # preprocess the images to transform from filenames to images to tensors
  images = [preprocess(Image.open(image)) for image in images_fp]

  # preprocess the texts to transform from text to tensors
  images = torch.tensor(np.stack(images)).cuda()
  text_tokens = clip.tokenize(["This is " + str(desc) for desc in texts]).cuda()

  # encode the inputs
  with torch.no_grad():
    images_z = model.encode_image(images).float()
    texts_z = model.encode_text(text_tokens).float()

  return images_z, texts_z

In [ ]:
import torch

def cosine_similarity(images_z: torch.Tensor, texts_z: torch.Tensor):
  # normalise the image and the text
  images_z /= images_z.norm(dim=-1, keepdim=True)
  texts_z /= texts_z.norm(dim=-1, keepdim=True)

  # evaluate the cosine similarity between the sets of features
  similarity = (texts_z @ images_z.T)

  return similarity.cpu()

In [ ]:
images_fp, texts, queries_fp = get_data(val_dataset)

# Drop same class instances
filtered_images_fp = []
filtered_texts = []
seen_labels = set()

# texts is a numpy array, convert to list or iterate directly
for i, label in enumerate(texts):
    if label not in seen_labels:
        seen_labels.add(label)
        filtered_images_fp.append(images_fp[i])
        filtered_texts.append(label)

images_fp = filtered_images_fp
texts = filtered_texts

images_z, texts_z = encode_data(model, preprocess, images_fp, texts)
queries_z, texts_z = encode_data(model, preprocess, queries_fp, texts)
similarity = cosine_similarity(images_z, texts_z)

print(similarity)

In [ ]:
import matplotlib.pyplot as plt
import torch
from PIL import Image

def visualise_similarity(similarity: torch.Tensor, images_fp: list[str], texts: list[str]):
  similarity = similarity.numpy()
  count = len(texts)

  # create a matplotlib figure object
  plt.figure(figsize=(18, 12))

  # show similarity scores
  plt.imshow(similarity, vmin=0.1, vmax=0.3)

  # update plot ticks
  plt.yticks(range(count), texts, fontsize=18)
  plt.xticks([])

  # visualise each image
  for i, image_fp in enumerate(images_fp):
    image = Image.open(image_fp).convert("RGB")
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

  # print the scores
  for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
      plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

  # update spines
  for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

  # change plot limits
  plt.xlim([-0.5, count - 0.5])
  plt.ylim([count + 0.5, -2])

  # set title
  plt.title("Cosine similarity between text and image features", size=20)


def visualise_query_vs_support(
    similarity: torch.Tensor,
    query_images_fp: list[str],
    support_images_fp: list[str],
):
    similarity = similarity.numpy()
    num_queries = len(query_images_fp)
    num_supports = len(support_images_fp)

    # Create a dynamic figure size based on matrix dimensions
    plt.figure(figsize=(num_supports * 2, num_queries * 2))
    # plt.figure(figsize=(18, 12))

    # 2. Show similarity heatmap matrix
    plt.imshow(similarity, vmin=0.1, vmax=0.3, cmap="viridis")

    # 3. Clear default ticks since we are replacing them with images
    plt.xticks([])
    plt.yticks([])

    # 4. Visualise Support Images along the Top/Bottom (X-axis)
    # Placed slightly above the matrix (Y limits depend on your matrix origin)
    for i, img_fp in enumerate(support_images_fp):
        image = Image.open(img_fp).convert("RGB")
        # extent: (left, right, bottom, top)
        plt.imshow(
            image, extent=(i - 0.4, i + 0.4, -1.4, -0.6), origin="upper"
        )

    # 5. Visualise Query Images along the Left (Y-axis)
    for j, img_fp in enumerate(query_images_fp):
        image = Image.open(img_fp).convert("RGB")
        # extent: (left, right, bottom, top) placed to the left of column 0
        plt.imshow(
            image, extent=(-1.4, -0.6, j + 0.4, j - 0.4), origin="upper"
        )

    # 6. Print the similarity scores inside the matrix cells
    for x in range(similarity.shape[1]):
        for y in range(similarity.shape[0]):
            plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

    # 7. Strip borders/spines
    for side in ["left", "top", "right", "bottom"]:
        plt.gca().spines[side].set_visible(False)

    # 8. Adjust limits to fit both the matrix and the external images
    plt.xlim([-1.6, num_supports - 0.5])
    plt.ylim([num_queries - 0.5, -1.6])

    plt.title("Similarity between Query and Support Images", size=16, pad=40)

# visualise_query_vs_support(similarity, images_fp, queries_fp)
visualise_similarity(similarity, images_fp, texts)

In [ ]:
def embed_dataset_classnames(dataset, templates: list[str] = ["a photo of a {}."]):
  # create the list of descriptions and tokenise them
  classnames = dataset.classes

  texts_z_views = []
  for template in templates:
    descriptions = [template.format(c) for c in classnames]
    text_tokens = clip.tokenize(descriptions).cuda()

    # get the normalised textual features
    with torch.no_grad():
        texts_z = model.encode_text(text_tokens).float()
        texts_z /= texts_z.norm(dim=-1, keepdim=True)
        texts_z_views.append(texts_z)

  # evaluate the mean representation
  texts_z = torch.stack(texts_z_views).mean(dim=0)

  # renormalise
  texts_z /= texts_z.norm(dim=-1, keepdim=True)

  return classnames, texts_z

In [ ]:
import matplotlib.pyplot as plt


def visualise_probabilities(
    images_fp: list[str], classnames: list[str], texts_p: torch.Tensor, k: int = 5
  ):
  topk_p, topk_labels = texts_p.cpu().topk(k, dim=-1)
  # create a matplotlib figure object
  plt.figure(figsize=(10, 10))

  for i, image_fp in enumerate(images_fp):
    # read the image
    image = Image.open(image_fp).convert("RGB")

    # visualise the image
    plt.subplot(4, 4, 2 * i + 1)
    plt.imshow(image)
    plt.axis("off")

    # visualise the probabilties for the image
    plt.subplot(4, 4, 2 * i + 2)
    y = np.arange(topk_p.shape[-1])
    plt.grid()
    plt.barh(y, topk_p[i])
    plt.gca().invert_yaxis()
    plt.gca().set_axisbelow(True)
    plt.yticks(y, [classnames[index] for index in topk_labels[i].numpy()])
    plt.xlabel("probability")

  plt.subplots_adjust(wspace=0.5)
  plt.show()

In [ ]:
# get the text descriptions and their embeddings
texts, texts_z = embed_dataset_classnames(
  test_dataset,
  templates=["a photo of a {}", "a low-res picture of a {}"]
)

# evalaute the softmax from the cosine similarities
similarity = cosine_similarity(texts_z, images_z)
texts_p = (100 * similarity).softmax(dim=-1)

# visualise the top-5 predictions
visualise_probabilities(images_fp, texts, texts_p, k=5)

## ADAPTATION

In [ ]:
import copy
import math
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0


def set_peft_seed(seed: int = 0) -> None:
    """Make the lab results more reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_peft_seed(SEED)
print("PEFT device:", DEVICE)


In [ ]:
@torch.no_grad()
def peft_encode_images(clip_model, images):
    """Encode images with frozen CLIP and L2-normalize the image features."""
    image_features = clip_model.encode_image(images).float()
    return F.normalize(image_features, dim=-1)


@torch.no_grad()
def peft_encode_text_tokens(clip_model, tokens):
    """Encode tokenized prompts with frozen CLIP and L2-normalize the text features."""
    text_features = clip_model.encode_text(tokens).float()
    return F.normalize(text_features, dim=-1)


def accuracy_from_logits(logits, labels):
    """Top-1 classification accuracy in percent."""
    predictions = logits.argmax(dim=1)
    return (predictions == labels).float().mean().item() * 100


def count_parameters(clip_model):
    """Return total and trainable parameter counts."""
    total = sum(parameter.numel() for parameter in clip_model.parameters())
    trainable = sum(parameter.numel() for parameter in clip_model.parameters() if parameter.requires_grad)
    return total, trainable


@torch.no_grad()
def evaluate_image_classifier(classifier, loader, name="classifier"):
    """Evaluate a classifier whose forward input is an image batch."""
    classifier.eval()
    total_correct = 0
    total_seen = 0

    for images, labels in tqdm(loader, desc=f"evaluate {name}", leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = classifier(images)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_seen += labels.numel()

    return 100.0 * total_correct / total_seen


@torch.no_grad()
def extract_clip_feature_dataset(clip_model, loader):
    """Pre-compute frozen CLIP image features for faster adapter/LoRA training."""
    all_features, all_labels = [], []
    for images, labels in tqdm(loader, desc="extract frozen CLIP features", leave=False):
        images = images.to(DEVICE)
        features = peft_encode_images(clip_model, images)
        all_features.append(features.cpu())
        all_labels.append(labels.cpu())
    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)

In [ ]:
peft_train_features, peft_train_labels = extract_clip_feature_dataset(model, train_loader)
peft_test_features, peft_test_labels = extract_clip_feature_dataset(model, test_loader)

peft_feature_dim = peft_train_features.shape[1]
peft_num_classes = len(test_dataset.classes)

print("Number of classes:", len(test_dataset.classes))
print("Classes:", test_dataset.classes)
print("Cached train feature shape:", peft_train_features.shape)
print("Cached test feature shape:", peft_test_features.shape)

In [ ]:
class FeatureLinearProbe(nn.Module):
    """A linear classifier trained on frozen CLIP image features."""

    def __init__(self, feature_dim, num_classes):
        super().__init__()
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, image_features):
        return self.classifier(image_features)


def train_on_cached_features(model, train_features, train_labels, test_features, test_labels, epochs=20, lr=1e-3, batch_size=256, name="feature-model"):
    """Train a small module using pre-computed frozen CLIP features."""
    model = model.to(DEVICE)
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(trainable_parameters, lr=lr, weight_decay=0.1)

    loss_fn = nn.CrossEntropyLoss()

    train_features = train_features.to(DEVICE)
    train_labels = train_labels.to(DEVICE)
    test_features = test_features.to(DEVICE)
    test_labels = test_labels.to(DEVICE)

    n = train_features.shape[0]
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(n, device=DEVICE)
        total_loss = 0.0

        for start in range(0, n, batch_size):
            indices = permutation[start:start + batch_size]
            logits = model(train_features[indices])
            loss = loss_fn(logits, train_labels[indices])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * indices.numel()

        model.eval()
        with torch.no_grad():
            accuracy = accuracy_from_logits(model(test_features), test_labels)
        print(f"{name} | epoch {epoch+1:02d}/{epochs} | loss {total_loss/n:.4f} | acc {accuracy:.2f}%")

    model.eval()
    with torch.no_grad():
        return accuracy_from_logits(model(test_features), test_labels)


linear_probe = FeatureLinearProbe(peft_feature_dim, peft_num_classes)
linear_accuracy = train_on_cached_features(
    linear_probe,
    peft_train_features,
    peft_train_labels,
    peft_test_features,
    peft_test_labels,
    epochs=200,
    lr=1e-2,
    batch_size=32,
    name="Linear probe",
)
_, linear_trainable = count_parameters(linear_probe)

print(f"method: Linear probe, accuracy: {linear_accuracy}, trainable_params: {linear_trainable}")
print("Trainable parameters:", linear_trainable)


In [ ]:
PEFT_PROMPT_TEMPLATES = {
    "plain": ["a photo of a {}."],
    "object": ["a photo of a {}.", "a blurry photo of a {}.", "a close-up photo of a {}."],
    "cifar10": ["a photo of a {}.", "a small low-resolution photo of a {}.", "a cropped photo of a {}."],
}

@torch.no_grad()
def build_text_classifier(classnames, templates, clip_model):
    """Build one normalized text prototype per class from one or more prompt templates."""
    class_features = []

    for classname in classnames:
        prompts = [template.format(classname) for template in templates]
        tokens = clip.tokenize(prompts).to(DEVICE)
        prompt_features = peft_encode_text_tokens(clip_model, tokens)

        # Average multiple prompt views for the same class, then normalize again.
        class_feature = F.normalize(prompt_features.mean(dim=0), dim=0)
        class_features.append(class_feature)

    return torch.stack(class_features, dim=0)

In [ ]:
class CLIPAdapterClassifier(nn.Module):
    """CLIP + bottleneck adapter classifier.

    Full path:
        images -> frozen CLIP image encoder -> adapter -> normalized feature -> text similarity logits

    Fast cached path:
        frozen image_features -> adapter -> normalized feature -> text similarity logits
    """

    def __init__(self, clip_model, text_features, feature_dim, reduction=4, alpha=0.2, logit_scale=100.0):
        super().__init__()
        self.clip_model = clip_model
        hidden_dim = max(1, feature_dim // reduction)

        # The adapter is the only trainable part.
        self.adapter = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feature_dim),
        )

        # Register a tensor into the model, but do not treat it as a trainable parameter.
        self.register_buffer("text_features", F.normalize(text_features.float(), dim=-1))
        self.alpha = alpha
        self.logit_scale = logit_scale

        # Make the intention explicit: CLIP is frozen, adapter is trainable.
        for parameter in self.clip_model.parameters():
            parameter.requires_grad = False

    def adapt_features(self, image_features):
        """Apply the adapter and residual mixing in feature space."""
        adapted_features = self.adapter(image_features)
        mixed_features = (1.0 - self.alpha) * image_features + self.alpha * adapted_features
        return F.normalize(mixed_features, dim=-1)

    def forward_from_features(self, image_features):
        """Fast path used when frozen CLIP features have already been cached."""
        image_features = F.normalize(image_features.float(), dim=-1)
        adapted = self.adapt_features(image_features)
        return self.logit_scale * adapted @ self.text_features.T

    def forward(self, images):
        """Full path used at inference time: images go through CLIP first."""
        image_features = peft_encode_images(self.clip_model, images)
        return self.forward_from_features(image_features)


class CachedFeatureWrapper(nn.Module):
    """A tiny wrapper so train_on_cached_features can call model(features)."""

    def __init__(self, clip_peft_classifier):
        super().__init__()
        self.clip_peft_classifier = clip_peft_classifier

    def forward(self, image_features):
        return self.clip_peft_classifier.forward_from_features(image_features)


# Changed from PEFT_PROMPT_TEMPLATES["plain"] to PEFT_PROMPT_TEMPLATES["object"]
base_text_features = build_text_classifier(test_dataset.classes, PEFT_PROMPT_TEMPLATES["object"], model)

clip_adapter = CLIPAdapterClassifier(
    clip_model=model,
    text_features=base_text_features,
    feature_dim=peft_feature_dim,
    reduction=4,
    alpha=0.2,
)

# Train only the adapter, using cached frozen CLIP features for speed.
adapter_feature_trainer = CachedFeatureWrapper(clip_adapter)
adapter_accuracy = train_on_cached_features(
    adapter_feature_trainer,
    peft_train_features,
    peft_train_labels,
    peft_test_features,
    peft_test_labels,
    epochs=100,
    lr=1e-3,
    name="CLIP+Adapter",
)
_, adapter_trainable = count_parameters(clip_adapter)
print(f"method: CLIP+Adapter, accuracy: {adapter_accuracy}, trainable_params: {adapter_trainable}")
print("Trainable parameters:", adapter_trainable)

### Using the Trained CLIP+Adapter Model for Inference

Now that the `CLIP+Adapter` model is trained, let's see how to use it to predict classes for new images. We will take a batch of images from the `val_loader` and get the top-5 predicted classes for each.

In [ ]:
# Set the model to evaluation mode
clip_adapter.eval()

# Get a batch of images from the validation loader
# We'll use the pre-computed text features (base_text_features) that were used to initialize the adapter
classnames = test_dataset.classes

# Take the first batch from the validation loader
# The val_loader yields full episodes, so we need to extract images.
# For simplicity, let's grab a few images directly from the test_loader
# to demonstrate classification with actual ground truth labels if needed.
# However, the previous cell already trained on peft_test_features, so let's use those.

# Let's take the first 5 test features and their corresponding labels
num_inference_samples = 5
sample_features = peft_test_features[:num_inference_samples]
sample_labels = peft_test_labels[:num_inference_samples]

print(f"Inferring on {num_inference_samples} cached test features.")

with torch.no_grad():
    # The forward_from_features method is used here because we are passing cached features
    logits = clip_adapter.forward_from_features(sample_features.to(DEVICE))
    probabilities = torch.softmax(logits, dim=-1)

    # Get top 5 predictions for each sample
    top_p, top_indices = torch.topk(probabilities, k=5, dim=-1)

    for i in range(num_inference_samples):
        true_label_idx = sample_labels[i].item()
        true_classname = classnames[true_label_idx] # Assuming classnames map to indices
        print(f"\nSample {i+1} (True Label: {true_classname}):")
        for j in range(5):
            predicted_class_idx = top_indices[i, j].item()
            predicted_classname = classnames[predicted_class_idx]
            probability = top_p[i, j].item()
            print(f"  Top {j+1}: {predicted_classname} (Probability: {probability:.4f})")


## Validation

In [ ]:
# Set the model to evaluation mode
clip_adapter.eval()

# We'll use the pre-computed text features (base_text_features) that were used to initialize the adapter
# classnames = test_dataset.classes # Not directly used for episodic prototype building here

print(f"Running episodic validation on {num_batches_val} episodes.")

all_episode_predictions = []

with torch.no_grad():

    for batch in tqdm(val_loader, total=num_batches_val, desc="[Val Episodic]"):
        # Extract episode data (val_loader yields batch_size=1, so we squeeze)
        s_imgs_paths = batch['support_imgs'][0] # List of image file paths
        s_labels = batch['support_labels'].squeeze(0).to(DEVICE) # Tensor of support labels
        q_imgs_paths = batch['query_imgs'][0] # List of image file paths
        episode_id = batch['episode_id'].item()

        # 1. Load and preprocess images
        s_images_tensors = torch.stack([preprocess(Image.open(fp)) for fp in s_imgs_paths]).to(DEVICE)
        q_images_tensors = torch.stack([preprocess(Image.open(fp)) for fp in q_imgs_paths]).to(DEVICE)

        # 2. Get adapted features for support and query images
        # peft_encode_images gets raw CLIP features, then clip_adapter.adapt_features applies the adapter
        s_clip_features = peft_encode_images(model, s_images_tensors) # Raw CLIP features
        s_adapted_features = clip_adapter.adapt_features(s_clip_features) # Adapted features

        q_clip_features = peft_encode_images(model, q_images_tensors)
        q_adapted_features = clip_adapter.adapt_features(q_clip_features)

        # 3. Build episode-specific prototypes from adapted support features
        # s_labels are the true class IDs (e.g., 0-249) for the support images within this episode.
        unique_episode_labels = torch.unique(s_labels)
        episode_prototypes = []

        for class_id_in_episode in unique_episode_labels:
            mask = (s_labels == class_id_in_episode)
            prototype = s_adapted_features[mask].mean(dim=0)
            episode_prototypes.append(prototype / prototype.norm()) # Normalize prototype
        episode_prototypes = torch.stack(episode_prototypes) # Shape: [num_unique_classes, feature_dim]

        # 4. Classify query images against episode prototypes
        # Calculate similarity between query adapted features and episode prototypes
        # Scale by 100.0 as CLIP does
        logits = 100.0 * (q_adapted_features @ episode_prototypes.T)
        predictions_indices = logits.argmax(dim=-1) # These are indices 0 to num_unique_classes-1

        # Map these prediction indices back to the actual class IDs from unique_episode_labels
        predicted_class_ids = [unique_episode_labels[idx.item()].item() for idx in predictions_indices]

        # 5. Store predictions for this episode
        for i, pred_label in enumerate(predicted_class_ids):
            all_episode_predictions.append({
                "episode_id": episode_id,
                "query_idx": i,
                "predicted_label": pred_label
            })

# After processing all episodes, save predictions to CSV
import pandas as pd
pd.DataFrame(all_episode_predictions).to_csv("val_predictions_adapter.csv", index=False)
print("Episodic predictions saved to val_predictions_adapter.csv")

In [ ]:
class CLIPLowRankUpdateClassifier(nn.Module):
    """CLIP + low-rank feature update classifier.

    This mirrors the LoRA idea using a low-rank residual update on frozen CLIP image features.
    """

    def __init__(self, clip_model, text_features, feature_dim, rank=8, alpha=16.0, logit_scale=100.):
        super().__init__()
        self.clip_model = clip_model
        self.rank = rank
        self.scaling = alpha / rank

        # Low-rank update: h -> h + (alpha/r) * up(down(h)).
        # down: d -> r, up: r -> d, with r much smaller than d.
        self.down = nn.Linear(feature_dim, rank, bias=False)
        self.up = nn.Linear(rank, feature_dim, bias=False)

        # Common LoRA initialization: one branch is zero so the initial model equals frozen CLIP.
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.up.weight)

        self.register_buffer("text_features", F.normalize(text_features.float(), dim=-1))
        self.logit_scale = logit_scale

        for parameter in self.clip_model.parameters():
            parameter.requires_grad = False

    def update_features(self, image_features):
        """Apply a low-rank residual update to frozen CLIP image features."""
        delta = self.up(self.down(image_features)) * self.scaling
        return F.normalize(image_features + delta, dim=-1)

    def forward_from_features(self, image_features):
        """Fast path used during training with cached frozen CLIP features."""
        image_features = F.normalize(image_features.float(), dim=-1)
        updated_features = self.update_features(image_features)
        return self.logit_scale * updated_features @ self.text_features.T

    def forward(self, images):
        """Full path: images -> frozen CLIP -> low-rank update -> logits."""
        image_features = peft_encode_images(self.clip_model, images)
        return self.forward_from_features(image_features)


clip_lora_style = CLIPLowRankUpdateClassifier(
    clip_model=model,
    text_features=base_text_features,
    feature_dim=peft_feature_dim,
    rank=8,
    alpha=16.0,
)

lora_feature_trainer = CachedFeatureWrapper(clip_lora_style)
lora_accuracy = train_on_cached_features(
    lora_feature_trainer,
    peft_train_features,
    peft_train_labels,
    peft_test_features,
    peft_test_labels,
    epochs=100,
    lr=1e-2,
    name="CLIP+LoRA-style update",
)
_, lora_trainable = count_parameters(clip_lora_style)
print(f"method: CLIP+LoRA-style update, accuracy: {lora_accuracy}, trainable_params: {lora_trainable}")
print("Trainable parameters:", lora_trainable)


## PREV WORK

In [ ]:
import os
import clip
import torch
import torchmetrics
import pandas as pd

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Define constants
LR = 1e-4 # Learning Rate
BATCH_SIZE = 32 # Number of images processed in one iteration
SHOTS = 5 # 5-way 5-shot
EPOCHS = 5 # Number of epochs

CKPT_PATH  = "best_student.pt"
CSV_PATH  = "./AMLdataset/release/train.csv"
IMGS_DIR  = "./AMLdataset/release/images/"
VAL_CSV   = "./AMLdataset/release/test_episodes_release.csv"

# Define controls
best_acc = 0.0 # Used to save best checkpoints

# Load the models
device = "cuda" if torch.cuda.is_available() else "cpu"
student, preprocess = clip.load('ViT-B/32', device, jit=False)
teacher, _ = clip.load('ViT-L/14', device, jit=False)
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

# Download the dataset
train_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=True,
    transform=preprocess
)

test_dataset = AMLDataset(
    csv_path=CSV_PATH,
    imgs_dir=IMGS_DIR,
    train=False,
    transform=preprocess
)

val_dataset = AMLValidation(
    csv_path=VAL_CSV,
    imgs_dir=IMGS_DIR,
    transform=preprocess
)

# Prepare the data
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_train = len(train_dataloader)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Standard way to randomize
    drop_last=True,     # Helpful for CLIP to keep matrix dimensions consistent
    pin_memory=True     # Helpful for GPU
)
num_batches_test = len(test_dataloader)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,       # Standard way to randomize
    drop_last=True      # Helpful for CLIP to keep matrix dimensions consistent
)
num_batches_val = len(val_dataloader)

all_class_texts = torch.cat([clip.tokenize(f"a photo of a {c}") for c in train_dataset.classes]).to(device)
NUM_CLASSES = len(train_dataset.classes)

# Prepare criterion and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in student.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.1
)

# Cosine schedule: smoothly decays LR to 0 over all training steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS * len(train_dataloader)
)

# Distillation temperature
# Higher T -> softer teacher distribution -> gentler supervision signal
KD_TEMPERATURE = 4.0
KD_ALPHA       = 1.0   # weight of distillation loss vs contrastive loss

In [ ]:
def convert_models_to_fp32(model):
    for p in model.parameters():
        p.data = p.data.float()
        if p.grad is not None:
            p.grad.data = p.grad.data.float()

In [ ]:
def train_one_epoch(epoch: int) -> float:
    student.train()
    epoch_loss = 0.0

    for images, class_ids in tqdm(train_dataloader, total=num_batches_train, desc=f"[Train] Epoch {epoch}"):
        images = images.to(device)

        # Build paired text tokens for this batch
        # Each image gets the text of its own class - this is what CLIP aligns
        texts = clip.tokenize([
            f"a photo of a {train_dataset.classes[i]}"
            for i in class_ids
        ]).to(device)

        optimizer.zero_grad()

        # Student forward
        # logits_per_image[i, j] = similarity(image_i, text_j) * scale
        # Ground truth: diagonal (image i matches text i)
        logits_per_image, logits_per_text = student(images, texts)

        ground_truth = torch.arange(len(images), dtype=torch.long, device=device)

        # Symmetric contrastive loss (standard CLIP objective)
        contrastive_loss = (
            criterion(logits_per_image, ground_truth) +
            criterion(logits_per_text,  ground_truth)
        ) / 2

        # Knowledge Distillation loss
        # Teacher produces a soft similarity distribution over the batch.
        # Student is trained to match it - transfers teacher's "uncertainty"
        # about near-duplicate classes rather than forcing hard 0/1 targets.
        with torch.no_grad():
            t_logits_img, t_logits_txt = teacher(images, texts)

        # Soft targets via temperature scaling
        soft_targets_img = torch.softmax(t_logits_img / KD_TEMPERATURE, dim=-1)
        soft_targets_txt = torch.softmax(t_logits_txt / KD_TEMPERATURE, dim=-1)

        # KL divergence: how far is student distribution from teacher's?
        kd_loss = (
            torch.nn.functional.kl_div(
                torch.log_softmax(logits_per_image / KD_TEMPERATURE, dim=-1),
                soft_targets_img, reduction='batchmean'
            ) +
            torch.nn.functional.kl_div(
                torch.log_softmax(logits_per_text / KD_TEMPERATURE, dim=-1),
                soft_targets_txt, reduction='batchmean'
            )
        ) / 2 * (KD_TEMPERATURE ** 2)  # T² rescaling keeps gradient magnitude stable

        total_loss = (1 - KD_ALPHA) * contrastive_loss + KD_ALPHA * kd_loss

        # total_loss = contrastive_loss

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        if device == "cpu":
            optimizer.step()
            # Update the learning rate
            scheduler.step()
        else:
            convert_models_to_fp32(student)
            optimizer.step()
            # Update the learning rate
            scheduler.step()
            clip.model.convert_weights(student)

        epoch_loss += total_loss.item()

    return epoch_loss / len(train_dataloader)

In [ ]:
def evaluate_test() -> float:
    """
    Linear probe evaluation: encode all test images, compute cosine similarity
    against all class text prototypes, pick argmax.
    This is the standard zero-shot CLIP evaluation style, applied to our classes.
    """
    student.eval()

    # Pre-compute text prototypes for all classes once
    with torch.no_grad():
        text_tokens = clip.tokenize([
            f"a photo of a {c}" for c in train_dataset.classes
        ]).to(device)
        # shape: [NUM_CLASSES, D]
        text_feats = student.encode_text(text_tokens)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

    correct = 0
    total   = 0

    with torch.no_grad():
        for images, labels in tqdm(test_dataloader, total=num_batches_test, desc="[Test]"):
            images = images.to(device)
            labels = labels.to(device)

            # shape: [B, D]
            img_feats = student.encode_image(images)
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)

            # Cosine similarity matrix: [B, NUM_CLASSES]
            # The 100x scale matches CLIP's internal logit scale
            sims = (img_feats @ text_feats.T) * 100.0

            preds = sims.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return correct / total

In [ ]:
def evaluate_val() -> dict:
    """
    Prototypical network inference over pre-defined episodes.
    No query labels available — returns predictions keyed by episode_id
    for submission instead of computing accuracy.
    """
    student = teacher
    student.eval()
    predictions = {}  # {episode_id: [pred_0, pred_1, ...]}

    with torch.no_grad():
        for batch in tqdm(val_dataloader, total=num_batches_val, desc="[Val]"):
            s_imgs   = batch['support_imgs'].squeeze(0).to(device)
            s_labels = batch['support_labels'].squeeze(0).to(device)
            q_imgs   = batch['query_imgs'].squeeze(0).to(device)
            episode_id = batch['episode_id'].item()

            # Encode
            s_feats = student.encode_image(s_imgs)
            s_feats = s_feats / s_feats.norm(dim=-1, keepdim=True)

            q_feats = student.encode_image(q_imgs)
            q_feats = q_feats / q_feats.norm(dim=-1, keepdim=True)

            # Build prototypes
            n_classes  = int(s_labels.max().item()) + 1
            embed_dim  = s_feats.shape[-1]
            prototypes = torch.zeros(n_classes, embed_dim, device=device, dtype=q_feats.dtype)

            for cls_id in range(n_classes):
                mask = (s_labels == cls_id)
                prototypes[cls_id] = s_feats[mask].mean(dim=0)

            prototypes = prototypes / prototypes.norm(dim=-1, keepdim=True)

            # Classify queries — local class indices (0..N-1)
            sims  = q_feats @ prototypes.T
            preds = sims.argmax(dim=-1).cpu().tolist()

            predictions[episode_id] = preds

    return predictions

In [ ]:
best_acc = 0.0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(epoch)

    test_acc   = evaluate_test()   # standard classification accuracy on held-out split

    print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}")

    # Checkpoint on test accuracy (best proxy you have without val labels)
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'test_acc': test_acc,
        }, CKPT_PATH)
        print(f"Checkpoint saved (test_acc={test_acc:.4f})")

# After training, run final inference and save predictions
val_preds = evaluate_val()
pd.DataFrame([
    {"episode_id": ep_id, "query_idx": i, "predicted_label": label}
    for ep_id, preds in val_preds.items()
    for i, label in enumerate(preds)
]).to_csv("val_predictions.csv", index=False)
print("Predictions saved to val_predictions.csv")

In [ ]:
def evaluate_episodic(model, dataloader, device):
    model.eval()
    all_episode_accs = []

    print("Running Episodic Evaluation...")
    with torch.no_grad():
        for episode in tqdm(dataloader):
            # Setup (batch_size is 1, so we squeeze)
            s_imgs = episode["support_imgs"].squeeze(0).to(device)   # [25, 3, 224, 224]
            s_labels = episode["support_labels"].squeeze(0).to(device)
            q_imgs = episode["query_imgs"].squeeze(0).to(device)     # [25, 3, 224, 224]

            # Get Features
            s_feats = model.encode_image(s_imgs)
            q_feats = model.encode_image(q_imgs)

            # Normalize
            s_feats /= s_feats.norm(dim=-1, keepdim=True)
            q_feats /= q_feats.norm(dim=-1, keepdim=True)

            # Create Class Prototypes (The "Average" for each of the 5 classes)
            unique_labels = torch.unique(s_labels) # These are your 5 classes for this episode
            prototypes = []
            for label in unique_labels:
                # Find all support images that belong to this specific label and average them
                class_prototype = s_feats[s_labels == label].mean(dim=0)
                prototypes.append(class_prototype / class_prototype.norm())

            prototypes = torch.stack(prototypes) # Shape: [5, 512]

            # Classify Queries against Prototypes
            # [25 queries, 512] @ [512, 5 prototypes] -> [25, 5] similarity matrix
            logits = 100.0 * q_feats @ prototypes.T
            preds = logits.argmax(dim=-1)

            # Accuracy Calculation
            # Note: Few-shot query labels are usually 0-4 (indices of the unique_labels)
            # If your dataset provides true labels for queries, use them here:
            if "query_labels" in episode:
                q_labels = episode["query_labels"].squeeze(0).to(device)
                # Map the true labels to 0-4 range to match our prototypes index
                label_to_idx = {val.item(): i for i, val in enumerate(unique_labels)}
                target_indices = torch.tensor([label_to_idx[l.item()] for l in q_labels], device=device)

                acc = (preds == target_indices).float().mean()
                all_episode_accs.append(acc)

        final_acc = torch.stack(all_episode_accs).mean()
        print(f"\nFinal Episodic Accuracy: {final_acc:.2%}")
        return final_acc


evaluate_episodic(student, val_dataloader, device)